# UPI Transaction Analytics — 2024
### End-to-End Analysis of 250,000 UPI Transactions Across India

**Author:** Madhan P  
**Tools:** Python · PostgreSQL · SQL · Power BI  
**Dataset:** 250,000 transactions | Jan–Dec 2024  

---

## Objective
Analyze UPI payment behavior across Indian states, banks, age groups, and network types —
and identify patterns that drive transaction failures and fraud risk.

## What this notebook covers
1. Setup & Libraries
2. Load Dataset
3. Exploratory Data Analysis (EDA)
   - Transaction volume & value
   - State-wise behavior
   - Peak transaction hours
   - Weekend vs Weekday patterns
   - Age group spending
   - Bank-wise distribution
   - Device & Network analysis
4. Fraud Analysis
5. Database Integration (PostgreSQL)
6. Summary & Key Findings

## 1. Setup & Libraries

Importing core libraries for data manipulation and visualization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns




## 2. Load Dataset

Loading 250,000 UPI transactions recorded across India in 2024.  
Each row represents one transaction with metadata on the sender, receiver, bank, device, network, and fraud status.

**Columns include:**
- `transaction id`, `timestamp`, `transaction type` (P2P / P2M / Recharge)
- `amount (INR)`, `transaction_status` (SUCCESS / FAILED)
- `sender_bank`, `receiver_bank`, `sender_state`
- `device_type`, `network_type`, `fraud_flag`
- `hour_of_day`, `day_of_week`, `is_weekend` (pre-engineered time features)

In [2]:
df = pd.read_csv("upi_transactions_2024.csv")

### 2.1 Dataset Preview

Displaying the full dataframe to confirm load and inspect structure.

In [15]:
df

,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


### 2.2 First 5 Rows

In [4]:
df.head()

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Age Group Distribution

Checking how different age groups participate in UPI transactions.

> **Insight:** The 26–35 age group dominates UPI usage — this is the digitally-native working population that grew up with smartphones and adopted UPI from its early days.

In [8]:
df['sender_age_group'].value_counts()

sender_age_group
26-35    99849
36-45    50456
46-55    24841
18-25    62345
56+      12509
Name: count, dtype: int64

### 3.2 State-wise Transaction Volume

Which Indian states are leading UPI adoption?

> **Insight:** Maharashtra leads with ~37,000 transactions — roughly 15% of total volume. The top 5 states (Maharashtra, UP, Karnataka, Tamil Nadu, Delhi) together account for ~59% of all transactions, reflecting India's urban-digital divide.

In [9]:
df['sender_state'].value_counts()

sender_state
Maharashtra       37427
Uttar Pradesh     30125
Karnataka         29756
Tamil Nadu        25367
Delhi             24870
Telangana         22435
Gujarat           20061
Andhra Pradesh    20006
Rajasthan         19981
West Bengal       19972
Name: count, dtype: int64

> **Note:** Maharashtra alone contributes ~10% of total UPI volume — driven by Mumbai's dense financial activity.

In [10]:
#Maharashtra alone contributes around ~10% of total UPI volume

### 3.3 Column Overview

In [11]:
df.columns

Index(['transaction id', 'timestamp', 'transaction type', 'merchant_category',
       'amount (INR)', 'transaction_status', 'sender_age_group',
       'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank',
       'device_type', 'network_type', 'fraud_flag', 'hour_of_day',
       'day_of_week', 'is_weekend'],
      dtype='object')

### 3.4 Total Transaction Value

Calculating the total INR value processed across all 250,000 transactions.

In [14]:
df['amount (INR)'].sum()

np.int64(327939009)

### 3.5 Transaction Success Rate

> **Insight:** 95% of all transactions succeed. The 4.9% failure rate is a key reliability metric — even a 1% improvement at India's UPI scale (over 10 billion monthly transactions nationally) represents hundreds of millions of recovered transactions.

In [17]:
df['transaction_status'].value_counts(normalize=True)

transaction_status
SUCCESS    0.950496
FAILED     0.049504
Name: proportion, dtype: float64

### 3.6 Peak Transaction Hours

Grouping transactions into time buckets to identify when Indians pay the most.

> **Insight:** Evening (19–23) and Afternoon (13–18) slots together drive 72% of all transactions. Mornings (0–6) are the quietest — only 4.6% of volume. This timing aligns with post-work shopping, food delivery, and bill payment behaviour.

In [23]:
bins = [0, 6, 12, 18, 24]
labels = ['0-6', '7-12', '13-18', '19-23']

df['hour_group'] = pd.cut(
    df['hour_of_day'],
    bins=bins,
    labels=labels,
    right=False
)

# Group it
df.groupby('hour_group')['transaction id'].count()

hour_group
0-6      11620
7-12     58162
13-18    88982
19-23    91236
Name: transaction id, dtype: int64

### 3.7 Weekend vs Weekday — Spending by Day

> **Insight:** Transaction volume is fairly consistent across all days — no dramatic weekend spike. However, Monday shows slightly higher value (₹47.8M), possibly due to rent, EMI, and bill payments queued over the weekend and processed on Monday morning.

In [24]:
df.groupby('day_of_week')['amount (INR)'].sum()

day_of_week
Friday       46332583
Monday       47882908
Saturday     45867936
Sunday       47572544
Thursday     46620985
Tuesday      46846390
Wednesday    46815663
Name: amount (INR), dtype: int64

### 3.8 State-wise Transaction Value (INR)

> **Insight:** Maharashtra generates the highest transaction value at ₹49M — nearly 15% of total value processed. Karnataka (home to Bengaluru's tech industry) ranks 3rd, reflecting higher average transaction sizes from tech workers.

In [25]:
df.groupby('sender_state')['amount (INR)'].sum().sort_values(ascending=False)

sender_state
Maharashtra       49043948
Uttar Pradesh     40035717
Karnataka         38451158
Tamil Nadu        33343518
Delhi             32689865
Telangana         29750930
Rajasthan         26730470
Gujarat           25988190
Andhra Pradesh    25952619
West Bengal       25952594
Name: amount (INR), dtype: int64

### 3.9 Average Spend by Age Group

> **Insight:** The 36–45 group spends the most on average (₹1,424/transaction) — they are established earners making larger utility, travel, and shopping payments. The 56+ group spends the least (₹1,187), suggesting lower UPI comfort or smaller transaction use-cases. This gap is a product opportunity for banks targeting senior UPI adoption.

In [26]:
df.groupby('sender_age_group')['amount (INR)'].mean()

sender_age_group
18-25    1194.545449
26-35    1326.285239
36-45    1424.041242
46-55    1333.129141
56+      1187.568631
Name: amount (INR), dtype: float64

### 3.10 Bank-wise Transaction Count

> **Insight:** SBI dominates with 62,693 transactions — nearly 2x HDFC (37,485). This reflects SBI's massive retail customer base across Tier 2 and 3 cities, where UPI is often the first digital payment method. Private banks like Axis and ICICI follow with roughly equal share.

In [30]:
df.groupby('sender_bank')['transaction id'].count().sort_values(ascending=False)

sender_bank
SBI         62693
HDFC        37485
ICICI       29769
IndusInd    25173
Axis        25042
PNB         24946
Yes Bank    24860
Kotak       20032
Name: transaction id, dtype: int64

### 3.11 Device Type Distribution

> **Insight:** Android powers 75% of all UPI transactions (187,777), followed by iOS at ~20% (49,613). Web-based UPI is a tiny 5% (12,610). This confirms UPI is a mobile-first product — any reliability or UI improvements must be Android-first.

In [32]:
df.groupby('device_type')['transaction id'].count()

device_type
Android    187777
Web         12610
iOS         49613
Name: transaction id, dtype: int64

### 3.12 Network Type vs Transaction Status

> **Insight:** 3G has the highest failure rate proportionally (~5.2%) vs 4G (~5.0%) and 5G (~4.9%). While the gap seems small, at India's UPI scale this represents millions of failed transactions for rural 3G users. This is a fintech accessibility concern — financial inclusion cannot be complete while network reliability remains a barrier.

In [33]:
df.groupby('network_type')['transaction_status'].value_counts()

network_type  transaction_status
3G            SUCCESS                11820
              FAILED                   651
4G            SUCCESS               142349
              FAILED                  7464
5G            SUCCESS                59543
              FAILED                  3039
WiFi          SUCCESS                23912
              FAILED                  1222
Name: count, dtype: int64

---
## 4. Fraud Analysis

Examining the distribution and patterns of fraudulent transactions in the dataset.

**Overall fraud rate: 0.19%** (480 out of 250,000 transactions)

While this seems small, at India's national UPI scale (10B+ monthly transactions), even 0.19% represents ~19 million fraudulent transactions per month. Understanding the risk profile is critical.

In [34]:
df['fraud_flag'].value_counts()

fraud_flag
0    249520
1       480
Name: count, dtype: int64

### 4.1 Fraud Rate by Time of Day

> **Insight:** The 0–6 AM window has the highest fraud rate (0.198%) — nearly 12% higher than the 7–12 AM window. Late-night large transfers should be flagged for additional authentication.

In [37]:
df.groupby('hour_group')['fraud_flag'].mean()

hour_group
0-6      0.001979
7-12     0.001771
13-18    0.001978
19-23    0.001951
Name: fraud_flag, dtype: float64

### 4.2 Fraud Rate by Transaction Amount Category

Segmenting transactions into amount buckets to find which range attracts the most fraud.

| Category | Range | Fraud Rate |
|----------|-------|------------|
| Low | ₹0–₹1,000 | 0.188% |
| Medium | ₹1,001–₹5,000 | 0.186% |
| Large | ₹5,001–₹20,000 | **0.304%** |
| Very Large | ₹20,000+ | 0.000% |

> **Key Insight:** Large transactions (₹5,000–₹20,000) show a **63% higher fraud rate** compared to low and medium buckets. Fraudsters deliberately target mid-range amounts to avoid automated high-value detection thresholds. Very large transactions (₹20,000+) show zero fraud — likely because they already trigger manual verification or 2FA by banks.
>
> **Recommendation:** Banks should implement step-up authentication (OTP + PIN) specifically for transactions between ₹5,000–₹20,000, not just for amounts above ₹20,000.

In [39]:
bins = [0, 1000, 5000, 20000, float('inf')]
labels = ['Low', 'Medium', 'Large', 'Very Large']

df['amount_category'] = pd.cut(
    df['amount (INR)'],
    bins=bins,
    labels=labels
)


In [40]:
df.groupby('amount_category')['fraud_flag'].mean()

amount_category
Low           0.001877
Medium        0.001855
Large         0.003040
Very Large    0.000000
Name: fraud_flag, dtype: float64

---
## 5. Database Integration (PostgreSQL)

Loading the cleaned DataFrame into PostgreSQL for SQL-based analysis.

The database connection string is loaded securely using environment variables — never hardcoded — following production security best practices.

```bash
# Install required packages
pip install python-dotenv sqlalchemy psycopg2
```

Create a `.env` file in your project root (never commit this to GitHub):
```
DB_USER=postgres
DB_PASSWORD=your_password_here
DB_HOST=localhost
DB_PORT=5432
DB_NAME=madhandb
```

In [3]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine

load_dotenv()  # reads .env file

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

df.to_sql(
    "upi_data",
    engine,
    if_exists="append",
    index=False
)

print("DataFrame inserted!")

DataFrame inserted!


---
## 6. Summary & Key Findings

| # | Finding | Business Implication |
|---|---------|---------------------|
| 1 | **₹32.7 Crore** total value across 250K transactions | Average ticket size ~₹1,312 per transaction |
| 2 | **95% success rate** overall | 4.9% failure rate = opportunity for infra improvement |
| 3 | **Maharashtra leads** with ₹49M in value | Urban metros drive UPI value, not just volume |
| 4 | **26–35 age group** = largest user base (99,849 users) | Core UPI demographic is young working professionals |
| 5 | **36–45 age group** spends most (avg ₹1,424/txn) | Mid-career users = highest revenue per transaction |
| 6 | **SBI leads** with 62,693 transactions | Public sector banks drive volume in Tier 2/3 cities |
| 7 | **Android = 75%** of all transactions | UPI is mobile-first; Android optimization is critical |
| 8 | **3G shows highest failure rate** | Rural financial inclusion needs network reliability |
| 9 | **Fraud peaks in ₹5K–₹20K range** (0.304%) | Step-up auth should target mid-range, not just large txns |
| 10 | **Evening (19–23) = peak hour** with 91,236 transactions | Post-work payment window drives the most volume |

---

*Full SQL queries → `/sql/upi_queries.sql`*  
*Power BI Dashboard → `/dashboard/upi_data.pbix`*  
*Fraud Detection ML Model → `upi_fraud_detection.ipynb`*